# Project 04: GAN Instability and Mode Collapse Diagnostics

Intentionally stress GAN training to trigger Observatory Pro warnings,
then inspect anomaly summaries and health logs.


## Dataset Source and Download Instructions

- Dataset: MNIST handwritten digits
- Official source: http://yann.lecun.com/exdb/mnist/
- Auto-download in this notebook: `torchvision.datasets.MNIST(download=True)`
- Manual fallback:
  1. Download from the official source.
  2. Place files in `./data/MNIST/raw/`.
  3. Re-run the notebook.


In [ ]:
%pip -q install "neurogebra[logging]" torch torchvision matplotlib


In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from neurogebra.logging.adaptive import AdaptiveLogger, AnomalyConfig
from neurogebra.logging.epoch_summary import EpochSummarizer
from neurogebra.logging.health_warnings import AutoHealthWarnings
from neurogebra.logging.logger import LogLevel, TrainingLogger
from neurogebra.logging.tiered_storage import TieredStorage

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
ds = datasets.MNIST(root=Path("./data"), train=True, download=True, transform=transform)
ds = Subset(ds, np.arange(8000))
loader = DataLoader(ds, batch_size=128, shuffle=True, drop_last=True)

class GNet(nn.Module):
    def __init__(self, latent=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent, 128),
            nn.ReLU(),
            nn.Linear(128, 784),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)

class DNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        h = torch.relu(self.fc1(x.view(x.size(0), -1)))
        out = torch.sigmoid(self.fc2(h))
        return out, h

G = GNet().to(device)
D = DNet().to(device)

# Intentionally aggressive LR to trigger warnings
opt_g = optim.Adam(G.parameters(), lr=1e-2)
opt_d = optim.Adam(D.parameters(), lr=1e-2)
criterion = nn.BCELoss()

base_logger = TrainingLogger(level=LogLevel.EXPERT)
storage = TieredStorage(base_dir="./logs_gan_diagnostics", write_debug=True)
base_logger.add_backend(storage)
logger = AdaptiveLogger(base_logger, config=AnomalyConfig(loss_spike_pct=25.0, gradient_spike_factor=3.0))

warnings_engine = AutoHealthWarnings()
summarizer = EpochSummarizer()

g_hist, d_hist = [], []
logger.on_train_start(total_epochs=5, total_samples=len(loader.dataset), batch_size=loader.batch_size)

for epoch in range(5):
    logger.on_epoch_start(epoch)
    g_total = 0.0
    d_total = 0.0

    for batch_idx, (real, _) in enumerate(loader):
        real = real.to(device)
        bs = real.size(0)
        ones = torch.ones(bs, 1, device=device)
        zeros = torch.zeros(bs, 1, device=device)

        # D update
        opt_d.zero_grad()
        real_out, real_h = D(real)
        d_real = criterion(real_out, ones)
        z = torch.randn(bs, 64, device=device)
        fake = G(z)
        fake_out, _ = D(fake.detach())
        d_fake = criterion(fake_out, zeros)
        d_loss = d_real + d_fake
        d_loss.backward()

        d_grad_norms = {}
        for n, p in D.named_parameters():
            if p.grad is not None:
                gn = float(p.grad.data.norm().item())
                d_grad_norms[f"D.{n}"] = gn
                logger.on_gradient_computed(f"D.{n}", gn)

        opt_d.step()

        # G update
        opt_g.zero_grad()
        z = torch.randn(bs, 64, device=device)
        fake = G(z)
        fake_out, _ = D(fake)
        g_loss = criterion(fake_out, ones)
        g_loss.backward()

        g_grad_norms = {}
        for n, p in G.named_parameters():
            if p.grad is not None:
                gn = float(p.grad.data.norm().item())
                g_grad_norms[f"G.{n}"] = gn
                logger.on_gradient_computed(f"G.{n}", gn)

        opt_g.step()

        grad_norms = {**d_grad_norms, **g_grad_norms}
        hidden_np = real_h.detach().cpu().numpy()
        zeros_pct = float((np.abs(hidden_np) < 1e-8).mean() * 100.0)
        sat_pct = float((np.abs(hidden_np) > 0.99).mean() * 100.0)

        alerts = warnings_engine.check_batch(
            epoch=epoch,
            batch=batch_idx,
            loss=float(g_loss.item()),
            gradient_norms=grad_norms,
            activation_stats={
                "D.hidden": {
                    "activation_type": "relu",
                    "zeros_pct": zeros_pct,
                    "saturation_pct": sat_pct,
                }
            },
        )
        for alert in alerts:
            logger.on_health_check(
                check_name=alert.rule_name,
                severity=alert.severity,
                message=alert.message,
                recommendations=alert.recommendations,
            )

        summarizer.record_batch(
            epoch=epoch,
            metrics={"g_loss": float(g_loss.item()), "d_loss": float(d_loss.item())},
            gradient_norms=grad_norms,
            activation_stats={"D.hidden": {"zeros_pct": zeros_pct, "saturation_pct": sat_pct}},
        )

        logger.on_batch_end(batch_idx, epoch=epoch, loss=float(g_loss.item()), metrics={"d_loss": float(d_loss.item())})
        g_total += float(g_loss.item())
        d_total += float(d_loss.item())

    avg_g = g_total / len(loader)
    avg_d = d_total / len(loader)
    g_hist.append(avg_g)
    d_hist.append(avg_d)

    print(summarizer.finalize_epoch(epoch).format_text())
    logger.on_epoch_end(epoch, metrics={"loss": avg_g, "val_loss": avg_d, "accuracy": 0.0, "val_accuracy": 0.0})

logger.on_train_end(final_metrics={"g_loss": g_hist[-1], "d_loss": d_hist[-1]})
storage.flush()
storage.close()

print("Anomaly summary:")
print(logger.get_anomaly_summary())
print("Total health warnings:", len(warnings_engine.warnings))
print("Tiered storage summary:", storage.summary())


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(g_hist, label="generator_loss")
plt.plot(d_hist, label="discriminator_loss")
plt.title("GAN Diagnostics Run")
plt.legend()
plt.show()
